# 3.3 메모리(Memory)

메모리는 에이전트가 과거 경험을 저장하고 검색하여 의사결정에 활용하는 역량이다.

### Perceive → Store → Retrieve → Plan → Reflect

생성형 에이전트 메모리의 핵심 사이클이다.

## 이론: 메모리 계층

### 3계층 메모리 구조

| 계층 | 용량 | 속도 | 용도 |
|---|---|---|---|
| 작업메모리 | 7±2개 | 빠름 | 현재 맥락 |
| 일화적메모리 | 중간 | 중간 | 구체적 사건 |
| 의미적메모리 | 큼 | 느림 | 일반 지식 |

### 메모리 검색 공식

```
검색점수 = 0.5 × 관련성 + 0.2 × 최신성 + 0.3 × 중요도
```

### 적응형 망각

중요하지 않고 오래된 정보는 자동으로 삭제된다.

## 환경 설정

In [1]:
from utils_openai import *
import time

heading("3.3 메모리(Memory)")
step_print(1, "API 준비", f"모델: {MODEL}")


────────────────────────────────────────
  3.3 메모리(Memory)
────────────────────────────────────────
  [Step 1] API 준비
    모델: gpt-4o-mini


## 실습 1: MemoryStream으로 저장 및 검색

In [2]:
step_print(1, "메모리 저장", "정보를 메모리에 저장한다")

# MemoryStream 생성
memory = MemoryStream()

# 여러 메모리 저장
memories_to_add = [
    ("파이썬은 데이터 과학에 많이 사용된다.", 0.8),
    ("사용자는 효율성을 중시한다.", 0.9),
    ("머신러닝 프로젝트가 진행 중이다.", 0.7),
    ("프로젝트 마감일이 다음 주다.", 0.85),
]

for content, importance in memories_to_add:
    memory.add(content, importance=importance)
    print(f"  ✓ 저장: {content[:40]}... (중요도: {importance})")

print(f"총 메모리: {len(memory)} 개")

  [Step 1] 메모리 저장
    정보를 메모리에 저장한다
  ✓ 저장: 파이썬은 데이터 과학에 많이 사용된다.... (중요도: 0.8)
  ✓ 저장: 사용자는 효율성을 중시한다.... (중요도: 0.9)
  ✓ 저장: 머신러닝 프로젝트가 진행 중이다.... (중요도: 0.7)
  ✓ 저장: 프로젝트 마감일이 다음 주다.... (중요도: 0.85)
총 메모리: 4 개


In [4]:
step_print(2, "메모리 검색", "관련성 + 최신성 + 중요도로 검색한다")

# 쿼리로 검색
query = "파이썬과 프로젝트"
retrieved = memory.retrieve(query, top_k=2)

print(f"쿼리: '{query}'")
print(f"검색된 메모리:")
for i, mem in enumerate(retrieved, 1):
    print(f"  {i}. {mem['content']}")

  [Step 2] 메모리 검색
    관련성 + 최신성 + 중요도로 검색한다
쿼리: '파이썬과 프로젝트'
검색된 메모리:
  1. 프로젝트 마감일이 다음 주다.
  2. 사용자는 효율성을 중시한다.


## 실습 2: 계층적 메모리 구조

In [5]:
step_print(3, "계층적 메모리", "3계층에서 정보를 분류한다")

# 계층별로 저장
working = []  # 작업메모리 (최근 7개)
episodic = []  # 일화적메모리 (중요 중간 정도)
semantic = []  # 의미적메모리 (매우 중요)

test_data = [
    ("지금 오후 2시다.", 0.3),  # 작업메모리
    ("어제 회의에서 결정한 사항이 있다.", 0.6),  # 일화적
    ("에이전트는 환경과 상호작용하며 학습한다.", 0.95),  # 의미적
]

for content, importance in test_data:
    if importance < 0.4:
        working.append(content)
        layer = "작업메모리"
    elif importance < 0.8:
        episodic.append(content)
        layer = "일화적메모리"
    else:
        semantic.append(content)
        layer = "의미적메모리"
    
    print(f"  → {layer}: {content[:35]}...")

kv_print({
    "작업메모리": len(working),
    "일화적메모리": len(episodic),
    "의미적메모리": len(semantic)
})

  [Step 3] 계층적 메모리
    3계층에서 정보를 분류한다
  → 작업메모리: 지금 오후 2시다....
  → 일화적메모리: 어제 회의에서 결정한 사항이 있다....
  → 의미적메모리: 에이전트는 환경과 상호작용하며 학습한다....
  작업메모리   1
  일화적메모리  1
  의미적메모리  1


## 실습 3: Perceive→Store→Retrieve→Plan→Reflect 사이클

In [6]:
heading("생성형 에이전트 메모리 사이클")

# Phase 1: Perceive - 관찰
step_print(1, "인지(Perceive)", "환경을 관찰한다")

observations = [
    "사용자가 강화학습에 관심이 있다.",
    "시스템 응답 속도가 중요하다.",
    "API 비용을 최소화해야 한다."
]

memory2 = MemoryStream()
for obs in observations:
    memory2.add(obs, importance=0.7)
    print(f"  ✓ 관찰: {obs}")


────────────────────────────────────────
  생성형 에이전트 메모리 사이클
────────────────────────────────────────
  [Step 1] 인지(Perceive)
    환경을 관찰한다
  ✓ 관찰: 사용자가 강화학습에 관심이 있다.
  ✓ 관찰: 시스템 응답 속도가 중요하다.
  ✓ 관찰: API 비용을 최소화해야 한다.


In [7]:
# Phase 2: Retrieve - 검색
step_print(2, "회상(Retrieve)", "관련 경험을 검색한다")

query = "강화학습 최적화"
retrieved2 = memory2.retrieve(query, top_k=2)

print(f"쿼리: '{query}'")
for mem in retrieved2:
    print(f"  → {mem['content']}")

  [Step 2] 회상(Retrieve)
    관련 경험을 검색한다
쿼리: '강화학습 최적화'
  → API 비용을 최소화해야 한다.
  → 시스템 응답 속도가 중요하다.


In [11]:
# Phase 3: Plan - 계획
step_print(3, "계획(Plan)", "메모리 기반 계획을 수립한다")

retrieved_text = "\n".join([m['content'] for m in retrieved2])

plan = ask(
    f"""현재 관찰:
{retrieved_text}

이를 바탕으로 3 단계 계획을 세우다.""",
    temperature=0.6,
    max_tokens=150
)

print(f"계획:{plan}")

  [Step 3] 계획(Plan)
    메모리 기반 계획을 수립한다
계획:API 비용을 최소화하고 시스템 응답 속도를 중요시하는 3단계 계획을 아래와 같이 제안합니다.

### 1단계: 현황 분석 및 최적화
- **API 사용 분석**: 현재 사용 중인 API의 호출 빈도와 비용을 분석합니다. 어떤 API가 가장 많은 비용을 발생시키는지, 어느 부분에서 응답 시간이 느린지 파악합니다.
- **불필요한 호출 제거**: 중복되거나 불필요한 API 호출을 제거하여 비용을 절감합니다. 예를 들어, 캐싱을 통해 자주 요청되는 데이터를 저장하고 재사용할 수 있습니다.
- **응답 시간 측정**: 각 API의


In [13]:
# Phase 4: Reflect - 반성
step_print(4, "반성(Reflect)", "경험으로부터 통찰을 도출한다")

insight = memory2.reflect(query)

print(f"도출된 통찰:{insight}")
print(f"(통찰이 의미적 메모리에 저장됨)")

  [Step 4] 반성(Reflect)
    경험으로부터 통찰을 도출한다
도출된 통찰:사용자가 강화학습에 관심을 가지고 있는 만큼, API 비용을 최소화하면서도 시스템의 응답 속도를 최적화하는 것이 중요하다. 이를 통해 사용자 경험을 향상시키고, 효율적인 자원 관리를 통해 비용 절감 효과를 동시에 달성할 수 있다.
(통찰이 의미적 메모리에 저장됨)


## 핵심 개념

### 3계층 메모리의 역할

1. **작업메모리**: "지금 이 순간"
2. **일화적메모리**: "그럴 때가 있었다"
3. **의미적메모리**: "일반적으로 알려진 사실"

### 검색 점수 계산

```python
score = 0.5 * relevance + 0.2 * recency + 0.3 * importance
```

### 생성형 에이전트 메모리 사이클

```
Perceive (관찰) → Store (저장) 
  ↓
Retrieve (검색) → Plan (계획)
  ↓
Reflect (반성) → 메모리 진화
```

이 사이클이 반복되며 에이전트는 점점 더 지능해진다.